
1. **Persiapan** — pasang library, atur `RANDOM_STATE`.
2. **Muat data** — unggah file xlsx/csv.
3. **Pembersihan rule-based** — kosong, duplikat, terlalu pendek, tanpa huruf
4. **Kamus & aturan** — kamus normalisasi, stopword, pengecualian stemming.
5. **Fungsi preprocessing** — case folding → cleaning → tokenization → normalization → stopword removal → stemming
6. **Terapkan ke seluruh data** — menghasilkan kolom `clean_text`.
7. **Filter relevansi** — buang komentar yang tidak menyinggung disabilitas/pekerjaan
8. **Cek kualitas & hapus duplikat `clean_text`**.
9. **Simpan `data_preprocessed.csv`**.

### Sel 1 — Persiapan library dan parameter
 memasang `PySastrawi` (stemmer bahasa Indonesia) dan `emoji`, mengimpor semua library, mengunduh daftar stopword NLTK, dan menetapkan `RANDOM_STATE = 42`.
**Mengapa:** `RANDOM_STATE` dipakai pada semua pengacakan (sampling audit, dll.) supaya hasil bisa diulang persis sama.
**Cek hasil:** harus tercetak `Library siap.`

In [ ]:
# Pasang library (tanda ! = perintah terminal di Colab)
!pip install -q PySastrawi emoji

# Library bawaan Python: regex, HTML, waktu, simpan model, penghitung kata
import re, html, time, joblib
from collections import Counter

# Library data & grafik
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# emoji: mengenali & menghapus emoji | nltk: stopword | Sastrawi: stemmer Indonesia
import emoji
import nltk
from tqdm.auto import tqdm
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
# scikit-learn (dipakai lebih jauh di notebook pemodelan)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Unduh daftar stopword NLTK (quiet = tanpa pesan panjang)
nltk.download('stopwords', quiet=True)
# Aktifkan progress bar untuk .progress_map()
tqdm.pandas()
# Tampilkan teks sampai 120 karakter di tabel
pd.set_option('display.max_colwidth', 120)

# Angka acak tetap agar hasil selalu sama setiap dijalankan
RANDOM_STATE = 42   # agar hasil split selalu sama (reproducible)
print("Library siap.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 20.9 MB/s eta 0:00:00
Library siap.


 lalu membaca file `.xlsx` atau `.csv` ke tabel `df`.
data berasal dari notebook EDA + pelabelan Gemini, jadi sudah memuat kolom `text` dan `label_annotator_1`.

In [ ]:
import pandas as pd
from google.colab import files

print("Pilih file data dari komputer Anda...")
# Munculkan tombol unggah file; hasilnya kamus {nama_file: isi}
uploaded = files.upload()
# Ambil nama file yang diunggah
nama_file = next(iter(uploaded))

try:
    # Pilih cara baca sesuai ekstensi file
    if nama_file.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(nama_file)
    else:
        df = pd.read_csv(nama_file)
    # Tampilkan konfirmasi & ukuran data
    print("Berhasil memuat data!")
    print(f"File        : {nama_file}")
    print(f"Ukuran data : {df.shape}")
    display(df.head())
except Exception as e:
    print(f"Gagal membaca file: {e}")

Pilih file data dari komputer Anda...


Saving data_untuk_labeling_full.xlsx to data_untuk_labeling_full.xlsx
Berhasil memuat data!
File        : data_untuk_labeling_full.xlsx
Ukuran data : (13613, 13)


,video_id,video_title,comment_id,is_reply,parent_id,author,text,likeCount,publishedAt,is_relevant_keyword,emoji_count,emoji_str,label_annotator_1
0,sfeRpL4t6yY,Menteri Risma Dikritik Masyarakat Usai Paksa Penyandang Tunarungu Lakukan Hal Ini | Kabar Petang,Ugy__Xbktb9UbUgv5M94AaABAg,False,NaN,@ourthings5977,dongo,0,2023-12-03T16:21:48Z,False,0,NaN,eksklusi
1,sfeRpL4t6yY,Menteri Risma Dikritik Masyarakat Usai Paksa Penyandang Tunarungu Lakukan Hal Ini | Kabar Petang,Ugxmuu_b9zZ0DU_Ptad4AaABAg,False,NaN,@Hellebæk-Ålsgårde_009,Geram gue liat ibu mentri menolak kodrat Allah.,0,2023-08-25T01:00:25Z,False,0,NaN,eksklusi
2,sfeRpL4t6yY,Menteri Risma Dikritik Masyarakat Usai Paksa Penyandang Tunarungu Lakukan Hal Ini | Kabar Petang,UgwGj1x5QwlQQHqiYO54AaABAg,False,NaN,@elmiutami1515,"Maklumi aja,,namanya juga bukan guru disabilitas ,,gk usah dipanjang2kan dn di plintir2kan ntar timbul masalah baru...",0,2023-02-16T05:41:07Z,True,0,NaN,eksklusi
3,HqYHDaPKWn8,Supir tunarungu mendapat penumpang spesial,UgycSjbhAcoghDks6Uh4AaABAg,False,NaN,@indahnganjuk9302,Teharu🙏 penupang,0,2025-10-08T11:28:36Z,False,1,🙏,inklusi
4,HqYHDaPKWn8,Supir tunarungu mendapat penumpang spesial,Ugy9MRkhly_nWufIUAx4AaABAg,False,NaN,@dsrcourse9407,"Bahasa terindah yg ingin ku pelajari, ya bahasa isyarat ❤",0,2025-10-06T11:08:06Z,True,1,❤,inklusi


### Eksplorasi data awal & pembersihan rule-based

mencetak jumlah baris/kolom, kolom yang punya nilai kosong, dan sebaran label mentah. Variabel `LABEL_COL` menentukan kolom label yang dipakai di seluruh notebook.


In [ ]:
# Nama kolom label. GANTI di sini bila memakai label hasil pelabelan ulang
LABEL_COL = 'label_annotator_1'

print("Jumlah baris  :", len(df))
print("Jumlah kolom  :", df.shape[1], "->", list(df.columns))
# Hanya tampilkan kolom yang punya nilai kosong (parent_id & emoji_str kosong itu normal)
print("\nMissing value per kolom:")
print(df.isna().sum()[lambda s: s > 0])
print("\nDistribusi label (data mentah):")
# Sebaran label: berapa inklusi vs eksklusi
print(df[LABEL_COL].value_counts(dropna=False))

Jumlah baris  : 13613
Jumlah kolom  : 13 -> ['video_id', 'video_title', 'comment_id', 'is_reply', 'parent_id', 'author', 'text', 'likeCount', 'publishedAt', 'is_relevant_keyword', 'emoji_count', 'emoji_str', 'label_annotator_1']

Missing value per kolom:
parent_id    12551
emoji_str     8796
dtype: int64

Distribusi label (data mentah):
label_annotator_1
eksklusi    7693
inklusi     5920
Name: count, dtype: int64


### Sel 4 — Pembersihan rule-based (funnel)
membuat salinan `df_cleaned`, lalu berturut-turut membuang (1) teks kosong, (2) duplikat persis, (3) komentar < 3 karakter, (4) komentar tanpa huruf. Tabel `funnel_df` mencatat berapa baris hilang di tiap langkah.

In [ ]:
# Kerja di salinan agar data asli df tetap utuh
df_cleaned = df.copy()
# Daftar pencatat jumlah baris di setiap tahap
funnel = [("Data mentah", len(df_cleaned))]

# Normalisasi minimal teks: unescape HTML + trim spasi
# Ubah entitas HTML (&amp; &#39;) jadi karakter biasa, lalu buang spasi awal/akhir
df_cleaned['text'] = df_cleaned['text'].astype('string').map(lambda t: html.unescape(t).strip() if pd.notna(t) else t)

# Aturan 1: buang teks kosong
# (1) Nilai kosong
df_cleaned = df_cleaned[df_cleaned['text'].notna() & (df_cleaned['text'] != '')]
funnel.append(("Setelah hapus missing value", len(df_cleaned)))

# Aturan 2: buang komentar yang persis sama (anti-spam & anti-bias frekuensi)
# (2) Duplikat persis
df_cleaned = df_cleaned.drop_duplicates(subset='text', keep='first')
funnel.append(("Setelah hapus duplikat", len(df_cleaned)))

# Aturan 3: buang komentar < 3 karakter (mis. 'ok', 'p')
# (3) Komentar < 3 karakter
df_cleaned = df_cleaned[df_cleaned['text'].str.len() >= 3]
funnel.append(("Setelah hapus komentar < 3 karakter", len(df_cleaned)))

# Aturan 4: buang komentar tanpa huruf sama sekali (hanya emoji/simbol/angka)
# (4) Komentar tanpa huruf (hanya emoji / simbol / angka)
df_cleaned = df_cleaned[df_cleaned['text'].str.contains(r'[A-Za-z]', regex=True)]
funnel.append(("Setelah hapus komentar tanpa huruf", len(df_cleaned)))

df_cleaned = df_cleaned.reset_index(drop=True)
df_cleaned['text'] = df_cleaned['text'].astype(str)

# Susun tabel funnel + kolom jumlah baris yang dihapus di tiap tahap
funnel_df = pd.DataFrame(funnel, columns=['Tahap', 'Jumlah baris'])
funnel_df['Dihapus'] = (-funnel_df['Jumlah baris'].diff()).fillna(0).astype(int)
funnel_df

,Tahap,Jumlah baris,Dihapus
0,Data mentah,13613,0
1,Setelah hapus missing value,13613,0
2,Setelah hapus duplikat,13613,0
3,Setelah hapus komentar < 3 karakter,13613,0
4,Setelah hapus komentar tanpa huruf,13613,0


### Preprocessing teks

### Sel 5 — Kamus normalisasi, stopword, dan pengecualian stemming
 menyiapkan "bahan" preprocessing:
- `NORM` — kamus kata tidak baku → baku (kamus alay dari GitHub + aturan buatan sendiri + koreksi).
- `STOP_BARU` — daftar stopword NLTK dikurangi kata negasi (`NEGASI`) dan kata bermakna (`KEEP_WORDS`), ditambah stopword khusus komentar YouTube.
- `STEM_EXC_BARU` — kata yang **tidak boleh** di-stem karena Sastrawi salah memotong (mis. *semoga*, *kesempatan*).


In [ ]:
# os & Sastrawi hanya untuk menemukan file kamus kata dasar Sastrawi
import os
import Sastrawi

# ---------- Kamus normalisasi (slang -> baku) ----------
# Alamat kamus slang→baku (colloquial-indonesian-lexicon) di GitHub
URL_KAMUS = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
# Unduh kamus; hapus entri 'slang' ganda
kamus = pd.read_csv(URL_KAMUS).drop_duplicates(subset='slang', keep='first')
# Jadikan kamus (dictionary): kata slang → kata baku
NORM_DASAR = dict(zip(kamus['slang'], kamus['formal']))

# Kamus kata dasar bawaan Sastrawi: dipakai untuk memperbaiki huruf memanjang & mengukur kualitas
# Kamus kata dasar Sastrawi: dipakai memperbaiki huruf memanjang & mengukur kualitas
ROOT_WORDS = set(open(os.path.join(os.path.dirname(Sastrawi.__file__), 'Stemmer', 'data', 'kata-dasar.txt')).read().split())

# Aturan dari versi awal Anda
# Aturan normalisasi buatan sendiri (versi awal): negasi, singkatan, istilah inti
NORM_CUSTOM_AWAL = {
    'gak': 'tidak', 'ga': 'tidak', 'nggak': 'tidak', 'ngga': 'tidak', 'enggak': 'tidak',
    'engga': 'tidak', 'gk': 'tidak', 'tdk': 'tidak', 'tak': 'tidak', 'kaga': 'tidak', 'kagak': 'tidak',
    'yg': 'yang', 'krn': 'karena', 'karna': 'karena', 'dgn': 'dengan', 'utk': 'untuk',
    'kalo': 'kalau', 'klo': 'kalau', 'tau': 'tahu', 'gue': 'saya', 'gw': 'saya', 'bgt': 'banget',
    'tuli': 'tuli', 'bisu': 'bisu', 'tunarungu': 'tunarungu', 'disabilitas': 'disabilitas',
    'difabel': 'difabel', 'kerja': 'kerja',
}
# Tambahan pada versi perbaikan
# Aturan tambahan (versi perbaikan): negasi daerah, ucapan terima kasih, ungkapan religius, slang lain
NORM_CUSTOM_BARU = {
    # negasi (termasuk bahasa daerah) -> 'tidak'
    'g': 'tidak', 'teu': 'tidak', 'henteu': 'tidak', 'ora': 'tidak', 'ndak': 'tidak', 'nda': 'tidak',
    'ndk': 'tidak', 'gaak': 'tidak', 'gpp': 'tidak apa', 'gabisa': 'tidak bisa', 'gaboleh': 'tidak boleh',
    'bukannya': 'bukan', 'bukankah': 'bukan', 'bukanlah': 'bukan', 'belumlah': 'belum',
    'janganlah': 'jangan', 'tidaklah': 'tidak', 'tidakkah': 'tidak',
    # ucapan terima kasih -> satu bentuk
    'terimakasih': 'terima kasih', 'trimakasih': 'terima kasih', 'makasih': 'terima kasih',
    'makasi': 'terima kasih', 'mksh': 'terima kasih', 'trims': 'terima kasih', 'nuhun': 'terima kasih',
    'terimaksih': 'terima kasih', 'haturnuhun': 'terima kasih', 'thanks': 'terima kasih', 'thank': 'terima kasih',
    # ungkapan religius -> satu bentuk
    'masyaallah': 'masyaallah', 'masyallah': 'masyaallah', 'masyaalloh': 'masyaallah',
    'masyaaallah': 'masyaallah', 'alhamdulilah': 'alhamdulillah', 'amiin': 'amin', 'aamiin': 'amin',
    # singkatan / slang lain
    'bpk': 'bapak', 'bp': 'bapak', 'gub': 'gubernur', 'gubenur': 'gubernur', 'mantab': 'mantap',
    'muantap': 'mantap', 'rame': 'ramai', 'vidio': 'video', 'difable': 'difabel', 'ijin': 'izin',
    'sodara': 'saudara', 'komen': 'komentar', 'lu': 'kamu', 'lo': 'kamu', 'aing': 'saya', 'urang': 'saya',
    'abdi': 'saya', 'pisan': 'sekali', 'ku': 'aku', 'si': 'si', 'd': 'di', 'ortu': 'orang tua',
}
# Koreksi hasil kamus yang keliru / kata yang belum ternormalisasi (REVISI)
# Koreksi untuk hasil kamus yang keliru (mis. 'pa' di konteks 'PA KDM' = bapak, bukan 'apa')
NORM_KOREKSI = {
    'pa': 'bapak',      # kamus lama: 'pa' -> 'apa' (salah, konteks "PA KDM")
    'rakyt': 'rakyat', 'klu': 'kalau', 'kya': 'kayak', 'syg': 'sayang',
}
# Gabungkan kamus. Urutan penting: yang belakang menimpa yang depan
NORM_AWAL = {**NORM_DASAR, **NORM_CUSTOM_AWAL}
# Kamus normalisasi FINAL yang dipakai di fungsi normalization()
NORM = {**NORM_AWAL, **NORM_CUSTOM_BARU, **NORM_KOREKSI}

# ---------- Stopword ----------
# Kata negasi yang TIDAK boleh dibuang sebagai stopword (membalik makna kalimat)
NEGASI = {'tidak', 'tak', 'bukan', 'belum', 'jangan', 'tanpa', 'kurang',
          'nggak', 'gak', 'ga', 'enggak', 'engga', 'tdk', 'gk'}
# Stopword tambahan khas komentar medsos (partikel, sapaan, tawa)
STOP_TAMBAHAN_AWAL = {'nya', 'ya', 'sih', 'nih', 'tuh', 'deh', 'dong', 'kok', 'lah', 'kan', 'mah', 'yah',
                      'kak', 'bang', 'mbak', 'kang', 'the', 'wkwk', 'wkwkwk', 'haha', 'hehe'}
# Stopword tambahan versi perbaikan (sapaan, bahasa Sunda, dll.)
STOP_TAMBAHAN_BARU = STOP_TAMBAHAN_AWAL | {'si', 'bapak', 'pak', 'bung', 'bro', 'aku', 'kamu',
                      'an', 'ber', 'ah', 'eh', 'oh', 'euy', 'atuh', 'lho', 'wae', 'nu', 'ti', 'aya',
                      'sareng', 'hayu', 'wr', 'wb'}

# Kata dalam daftar NLTK yang SENGAJA dipertahaman karena membawa makna untuk inklusi/eksklusi
# Kata yang ada di daftar NLTK tetapi SENGAJA dipertahankan karena membawa makna inklusi/eksklusi
# (kemampuan 'bisa/boleh/mampu', evaluasi 'baik/benar', keraguan 'apakah/mungkin', 'mereka/kita', kontras 'tapi/padahal')
KEEP_WORDS = {
    # modalitas / kemampuan  ("tidak BISA", "tidak BOLEH", "MAMPU bekerja")
    'bisa', 'bisakah', 'boleh', 'bolehkah', 'bolehlah', 'dapat', 'mampu', 'mampukah', 'semampu', 'harus',
    'haruslah', 'harusnya', 'seharusnya', 'sebaiknya', 'sepantasnya', 'pantas', 'perlu', 'perlukah', 'usah',
    'percuma', 'mau', 'ingin', 'inginkah', 'hendak',
    # evaluasi  ("LUAR BIASA", "BAIK", "BENAR")
    'baik', 'sebaik', 'benar', 'betul', 'benarkah', 'biasa', 'luar', 'penting', 'pentingnya',
    'keterlaluan', 'terlalu', 'cukup', 'jelas',
    # pekerjaan (topik utama penelitian)
    'bekerja', 'mengerjakan', 'dikerjakan',
    # kata tanya / keraguan
    'apa', 'apakah', 'apaan', 'kapan', 'kenapa', 'mengapa', 'bagaimana', 'bagaimanakah', 'mana',
    'siapa', 'berapa', 'mungkin', 'masih',
    # pronomina kelompok (kita vs mereka)
    'mereka', 'kita', 'kami', 'kalian',
    # dukungan singkat (REVISI: sebelumnya 'siap' dan 'lanjut' terbuang sebagai stopword)
    'siap', 'lanjut',
    # konjungsi kontras
    'tapi', 'tetapi', 'namun', 'padahal', 'walau', 'walaupun', 'meski', 'meskipun', 'hanya', 'cuma', 'malah', 'justru',
}
# Daftar stopword bahasa Indonesia dari NLTK
NLTK_STOP = set(stopwords.words('indonesian'))
# Stopword versi awal (hanya untuk perbandingan)
STOP_AWAL = (NLTK_STOP | STOP_TAMBAHAN_AWAL) - NEGASI
# Stopword FINAL = NLTK + tambahan, dikurangi negasi dan kata bermakna
STOP_BARU = (NLTK_STOP | STOP_TAMBAHAN_BARU) - NEGASI - KEEP_WORDS

# Opsional: buang nama diri yang spesifik video (kdm, hasni, matahari, dst.). Default MATI,
# karena pada evaluasi di bawah menghapusnya justru menurunkan skor.
# Saklar: True = buang nama khusus video (kdm, hasni, ...). Default mati karena menurunkan skor pada evaluasi sebelumnya
HAPUS_ENTITAS_VIDEO = False
# Daftar nama diri/entitas khusus video
ENTITAS_VIDEO = {'kdm', 'dedi', 'dedy', 'mulyadi', 'hasni', 'hasna', 'zidan', 'steven', 'adel', 'sanjaya',
                 'ivan', 'udin', 'koko', 'matahari', 'store', 'dept', 'dep', 'department', 'depstore'}

# ---------- Stemmer ----------
# Kata yang dikecualikan dari stemming (versi awal)
STEM_EXC_AWAL = {'setuju', 'kesempatan', 'perusahaan', 'penyandang', 'menangis', 'bali', 'pisan',
                 'tuli', 'bisu', 'tunarungu', 'disabilitas', 'difabel', 'diskriminasi'}
# Pengecualian stemming FINAL: kata yang selalu salah dipotong Sastrawi
STEM_EXC_BARU = STEM_EXC_AWAL | {
    'kekurangan', 'kelebihan', 'keterbatasan', 'pemerintah', 'lapangan',
    'bekasi', 'pakuan', 'pengunjung', 'menyinari', 'mengasih',
    # REVISI: Sastrawi salah memotong
    'semoga', 'meninggal', 'idaman', 'sekedar', 'sekadar', 'mentri', 'kusus', 'berbasis',
    'eksis', 'rasis', 'gelis', 'keterbasan', 'didikan', 'berasa', 'perasaan',
    'pembelinya', 'berkenalan', 'lainya', 'anaku', 'memilah', 'perawakan',
    'terkesan', 'mengakui', 'ajakan'
}
# Buat objek stemmer Sastrawi
_stemmer = StemmerFactory().create_stemmer()
# Cache hasil stem agar kata yang sama tidak diproses ulang (jauh lebih cepat)
_stem_cache = {}

# Ringkasan ukuran kamus untuk pengecekan
print(f"Kamus: {len(NORM):,} entri | Stopword awal: {len(STOP_AWAL)} -> baru: {len(STOP_BARU)} "
      f"({len(KEEP_WORDS)} kata dipertahaman) | Kata dasar: {len(ROOT_WORDS):,}")

Kamus: 4,387 entri | Stopword awal: 766 -> baru: 713 (71 kata dipertahaman) | Kata dasar: 29,932


### Sel 6 — Fungsi preprocessing (6 tahap)
 mendefinisikan fungsi untuk tiap tahap
1. `case_folding` — huruf kecil + merapikan huruf unicode aneh (small caps, huruf tebal).
2. `cleaning` — buang URL, mention, emoji, angka, tanda baca, huruf berulang ("mantaaap" → "mantap"); "tuna rungu" disatukan jadi "tunarungu".
3. `tokenization` — pecah teks menjadi kata.
4. `normalization` — ubah kata tidak baku ke baku lewat kamus.
5. `stopword_removal` — buang stopword dan kata 1 huruf.
6. `stemming` — ubah ke kata dasar (dengan cache supaya cepat).


In [ ]:
# unicodedata: normalisasi huruf unicode | re: regex | emoji: hapus emoji
import unicodedata
import re
import emoji

# Huruf 'small caps' (mis. "ᴘᴀs ɴɢᴏᴍᴏɴɢ") -> huruf biasa. Tanpa ini komentar jadi kosong setelah cleaning.
# Tabel pengganti huruf 'small caps' (ᴘᴀs) ke huruf biasa; tanpa ini komentar jadi kosong setelah cleaning
_SMALLCAP = str.maketrans('ᴀʙᴄᴅᴇꜰɢʜɪᴊᴋʟᴍɴᴏᴘǫʀꜱᴛᴜᴠᴡʏᴢ', 'abcdefghijklmnopqrstuvwyz')

# Seragamkan jenis huruf (tebal, miring, fullwidth) menjadi huruf biasa
def normalisasi_unicode(text):
    text = text.translate(_SMALLCAP)
    return unicodedata.normalize('NFKC', text)   # juga mengubah huruf tebal/miring/fullwidth ke huruf biasa

# 1) Case folding
# TAHAP 1 — Case folding: semua huruf menjadi huruf kecil
def case_folding(text):
    return normalisasi_unicode(text).lower()

# 2) Cleaning
# TAHAP 2 — Cleaning: buang elemen yang tidak bermakna untuk persepsi
def cleaning(text):
    # Hapus karakter tak kasat mata
    text = re.sub(r'[\u200b-\u200d\ufeff]', '', text)                  # karakter zero-width
    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)                  # URL
    # Hapus mention (@username); handle YouTube = nama + 4 angka
    text = re.sub(r'@(?:[\w.\-]*?\d{4}|[\w.\-]+)', ' ', text)           # mention (handle YouTube: nama + 4 digit)
    # Hapus emoji
    text = emoji.replace_emoji(text, replace=' ')                       # emoji
    # Samakan penulisan: 'tuna rungu' / 'tuna-rungu' → 'tunarungu' (juga tunawicara, dst.)
    text = re.sub(r'\btuna\s*[- ]?\s*(rungu|runggu|wicara|wucara|netra|grahita)\b', r'tuna\1', text)  # tuna rungu -> tunarungu
    # Hapus angka dan semua tanda baca (sisakan huruf a-z dan spasi)
    text = re.sub(r'[^a-z\s]', ' ', text)                               # angka & tanda baca
    # Huruf berulang ≥ 3 jadi 1 ('mantaaap' → 'mantap')
    text = re.sub(r'(.)\1{2,}', r'\1', text)                            # "mantaaap" -> "mantap"
    return re.sub(r'\s+', ' ', text).strip()

# 3) Tokenization
# TAHAP 3 — Tokenization: pecah kalimat menjadi daftar kata
def tokenization(text):
    return text.split()

# 4) Normalization
# Cari kata di kamus NORM (dua putaran agar rantai g → enggak → tidak ikut terpecah)
def _norm_token(w):
    for _ in range(2):                       # dua putaran: g -> enggak -> tidak
        v = NORM.get(w, w)
        if v == w:
            break
        w = v
    return w

# Perbaiki huruf ganda salah ketik ('aamiin' → 'amin') HANYA bila hasilnya kata dasar valid
def _fix_elongation(w):
    # "aamiin" -> "amin", "cantikk" -> "cantik" bila hasilnya kata dasar yang valid
    if re.search(r'(.)\1', w):
        c = re.sub(r'(.)\1+', r'\1', w)
        if c in ROOT_WORDS:
            return c
    return w

# TAHAP 4 — Normalization: kata tidak baku → baku; frasa hasil normalisasi dipecah lagi menjadi token
def normalization(tokens):
    out = []
    for w in tokens:
        v = _norm_token(w)
        if v == w and w not in ROOT_WORDS:
            f = _fix_elongation(w)
            if f != w:
                v = _norm_token(f)
        out.extend(_norm_token(p) for p in v.replace('-', ' ').split())   # frasa -> beberapa token
    return out

# 5) Stopword removal (juga membuang token 1 huruf)
# TAHAP 5 — Stopword removal: buang kata umum & token 1 huruf
def stopword_removal(tokens):
    stop = STOP_BARU | (ENTITAS_VIDEO if HAPUS_ENTITAS_VIDEO else set())
    return [w for w in tokens if w not in stop and len(w) > 1]

# 6) Stemming (cache + pengecualian, termasuk bentuk berakhiran -nya/-lah/-kah/-pun)
# Stem satu kata: lewati bila ada di daftar pengecualian; lepas akhiran -nya/-lah/-kah/-pun dulu; pakai cache
def stem_token(w, exc=None):
    exc = STEM_EXC_BARU if exc is None else exc
    if w in exc:
        return w
    base = re.sub(r'(nya|lah|kah|pun)$', '', w)
    if base in exc:
        return base
    if w not in _stem_cache:
        _stem_cache[w] = _stemmer.stem(w)
    return _stem_cache[w]

# TAHAP 6 — Stemming: kata berimbuhan → kata dasar ('bekerja' → 'kerja')
def stemming(tokens):
    return [w for w in (stem_token(t) for t in tokens) if len(w) > 1]

# Rangkaian lengkap 6 tahap. Fungsi ini juga dipakai di aplikasi Streamlit
def proses_baru(text):
    return stemming(stopword_removal(normalization(tokenization(cleaning(case_folding(text))))))

### Sel 7 — Demonstrasi preprocessing pada 5 contoh
 menjalankan tiap tahap pada 5 kalimat contoh dan menampilkan hasil per tahap dalam satu tabel.



In [ ]:
# 5 kalimat uji
contoh = [
    "Setuju banget!!     😊👍 cek https://youtu.be/xxxx ya",
    "Tidak semua orang tuli sulit bekerja",
    "Tuna Rungu bukan berarti gak bisa kerja, yg penting krn mereka bekerja keras!!! @budi",
    "Luar biasa❤",
    "@Pepen8298Cerdas mantaaap amiiin, makasih ya",
]
rows = []
# Jalankan tahap demi tahap dan simpan hasil setiap tahap
for k in contoh:
    s1 = case_folding(k); s2 = cleaning(s1); s3 = tokenization(s2); s4 = normalization(s3)
    s5 = stopword_removal(s4); s6 = stemming(s5)
    rows.append({'asli': k, '1 case folding': s1, '2 cleaning': s2, '3 tokenization': s3,
                 '4 normalization': s4, '5 stopword': s5, '6 stemming': s6})
# Tampilkan sebagai tabel (.T = baris↔kolom agar muat di layar)
pd.DataFrame(rows).T

,0,1,2,3,4
asli,Setuju banget!! 😊👍 cek https://youtu.be/xxxx ya,Tidak semua orang tuli sulit bekerja,"Tuna Rungu bukan berarti gak bisa kerja, yg penting krn mereka bekerja keras!!! @budi",Luar biasa❤,"@Pepen8298Cerdas mantaaap amiiin, makasih ya"
1 case folding,setuju banget!! 😊👍 cek https://youtu.be/xxxx ya,tidak semua orang tuli sulit bekerja,"tuna rungu bukan berarti gak bisa kerja, yg penting krn mereka bekerja keras!!! @budi",luar biasa❤,"@pepen8298cerdas mantaaap amiiin, makasih ya"
2 cleaning,setuju banget cek ya,tidak semua orang tuli sulit bekerja,tunarungu bukan berarti gak bisa kerja yg penting krn mereka bekerja keras,luar biasa,cerdas mantap amin makasih ya
3 tokenization,"[setuju, banget, cek, ya]","[tidak, semua, orang, tuli, sulit, bekerja]","[tunarungu, bukan, berarti, gak, bisa, kerja, yg, penting, krn, mereka, bekerja, keras]","[luar, biasa]","[cerdas, mantap, amin, makasih, ya]"
4 normalization,"[setuju, banget, cek, ya]","[tidak, semua, orang, tuli, sulit, bekerja]","[tunarungu, bukan, berarti, tidak, bisa, kerja, yang, penting, karena, mereka, bekerja, keras]","[luar, biasa]","[cerdas, mantap, amin, terima, kasih, ya]"
5 stopword,"[setuju, banget, cek]","[tidak, orang, tuli, sulit, bekerja]","[tunarungu, bukan, tidak, bisa, kerja, penting, mereka, bekerja, keras]","[luar, biasa]","[cerdas, mantap, amin, terima, kasih]"
6 stemming,"[setuju, banget, cek]","[tidak, orang, tuli, sulit, kerja]","[tunarungu, bukan, tidak, bisa, kerja, penting, mereka, kerja, keras]","[luar, biasa]","[cerdas, mantap, amin, terima, kasih]"


### Penerapan dataset

### Sel 8 — Terapkan preprocessing ke seluruh data
 menjalankan tiap tahap pada seluruh komentar dan menyimpan hasil **setiap tahap** sebagai kolom terpisah (`text_casefold`, `text_clean`, `tokens`, `tokens_norm`, `tokens_stop`, `tokens_stem`), lalu menyatukannya menjadi `clean_text`.

In [ ]:
# Catat waktu mulai
t0 = time.time()
# Tahap 1: huruf kecil
df_cleaned['text_casefold'] = df_cleaned['text'].map(case_folding)
# Tahap 2: cleaning
df_cleaned['text_clean']    = df_cleaned['text_casefold'].map(cleaning)
# Tahap 3: tokenisasi
df_cleaned['tokens']        = df_cleaned['text_clean'].map(tokenization)
# Tahap 4: normalisasi
df_cleaned['tokens_norm']   = df_cleaned['tokens'].map(normalization)
# Tahap 5: stopword
df_cleaned['tokens_stop']   = df_cleaned['tokens_norm'].map(stopword_removal)
# Tahap 6: stemming (dengan progress bar)
df_cleaned['tokens_stem']   = df_cleaned['tokens_stop'].progress_map(stemming)
# Gabungkan token jadi satu kalimat bersih; ini yang masuk ke TF-IDF
df_cleaned['clean_text']    = df_cleaned['tokens_stem'].map(' '.join)
print(f"Selesai dalam {time.time() - t0:.1f} detik")
# Tampilkan 10 contoh acak sebelum → sesudah
df_cleaned[['text', 'clean_text', LABEL_COL]].sample(10, random_state=RANDOM_STATE)

  0%|          | 0/13613 [00:00<?, ?it/s]

Selesai dalam 8.2 detik


,text,clean_text,label_annotator_1
6844,Semoga Matahari terus berjaya,semoga matahari jaya,eksklusi
12808,"Mereka calon istri idaman,dibalik kekurangannya mempunyai semangat pekerja keras,,",mereka calon istri idaman balik kekurangan semangat kerja keras,inklusi
8604,"Jadii film drachin yg bos minimarket yang hebat ituuu ada versi real live nyaa,,,😌 difabel direkrut lgsg rame pembel...",film drachin bos minimarket hebat versi real live difabel rekrut langsung ramai beli,inklusi
99,mereka berisik di keheningan,mereka berisik hening,inklusi
3780,klu gubernur syg rakyt ya kya gini.,gubernur sayang rakyat kayak,eksklusi
3024,"MATAHARI KALIAN THE BEST , MEMPEKERJAKAN ORANG DIFABEL DENGAN BAIK , GOOD JOBB",matahari kalian best kerja orang difabel baik good jobb,inklusi
10310,Matahari Department store 👍,matahari department store,inklusi
7313,"Alhamdulillah,,,akhirnya di undang pula KDM,, 👍👍👍👍",alhamdulillah undang kdm,inklusi
6479,"ASSALAMUALAIKUM PA KDM,SEMOGA SEHAT ,KALO DI PANJANGKAN UMUR INSYA ALLAH INGIN KETEMU PENGEN SILATUROHIM😊❤",assalamualaikum kdm semoga sehat panjang umur insya allah ingin ketemu pengin silaturohim,eksklusi
1512,"PAK DEDY COBA CEK"" ADA NENEK NENEK MENINGGAL DI PUKULI GARA GARA MENCURI 2 BIJI LABU BUAT MAKAN""😭DI CIANJUR",dedy coba cek nenek nenek meninggal pukul gara gara curi biji labu makan cianjur,eksklusi


### Sel 9 — Buang komentar yang kosong setelah preprocessing
 mencari komentar yang `clean_text`-nya kosong (mis. "Wkwkwk", "pertama", hanya mention), mencetak labelnya, lalu membuangnya menjadi `df_final`.


In [ ]:
# Penanda komentar yang kosong setelah preprocessing
kosong = df_cleaned['clean_text'].str.strip() == ''
# Cetak jumlah & label komentar kosong
print(f"Komentar kosong setelah preprocessing: {kosong.sum()} baris "
      f"(label: {df_cleaned.loc[kosong, LABEL_COL].value_counts().to_dict()})")
print(df_cleaned.loc[kosong, ['text', LABEL_COL]].head(10))

# Buang komentar kosong; hasilnya df_final
df_final = df_cleaned.loc[~kosong].reset_index(drop=True)
print("\nJumlah data final:", len(df_final))
# Sebaran label setelah pembuangan
print(df_final[LABEL_COL].value_counts())

Komentar kosong setelah preprocessing: 48 baris (label: {'eksklusi': 46, 'inklusi': 2})
                                                            text  \
202                                                     Kedua🎉🎉🎉   
203                                                      pertama   
428   https://youtube.com/shorts/K9Elq0Hwz0Y?si=zkqOMuvOcnlTlGAs   
670                                           @spica30-b2w waduh   
783                                           @ainawaff235 🥹❤️‍🩹   
1520                                                   Wkwkwk😂😂😂   
2933                                                      Wkwkwk   
3256                                            Bapak aiiing❤❤❤❤   
3498                                                        Aing   
3516                                 ​@AdeSunarya-w4gterimakasih   

     label_annotator_1  
202           eksklusi  
203           eksklusi  
428           eksklusi  
670           eksklusi  
783            inklusi  
1520         

### Filter relevansi — buang komentar yang tidak relevan dengan peluang kerja

Komentar yang **tidak menyinggung penyandang disabilitas/tunarungu, pekerjaan/kesempatan kerja mereka, maupun perlakuan terhadap mereka** (mis. absen daerah *"Garut hadir"*, menanyakan alamat toko, pujian untuk toko atau pejabat) **tidak ikut diklasifikasikan**.
Komentar seperti itu tidak menunjukkan sikap inklusi maupun eksklusi, sehingga jika dipaksa masuk salah satu kelas, model justru belajar membedakan *"membahas topik atau tidak"*, bukan persepsi.

Filter berbasis **leksikon kata kunci** pada teks asli (sebelum stemming) dan diterapkan **sama untuk kedua kelas** (tidak boleh hanya pada satu kelas, karena akan menciptakan pola palsu).
Filter ini heuristik, bukan sempurna, karena itu sebagian sampel diekspor untuk **audit manual** (sel di bagian akhir notebook menghitung ketepatannya).

### Sel 10 — Filter relevansi (+ sampel audit)
 menandai komentar sebagai *relevan* bila teks aslinya memuat salah satu kata kunci (`POLA_RELEVAN`: tuli, tunarungu, kerja, karyawan, kesempatan, ramah, hasna, hasni, kembar, dst.). Yang tidak cocok dibuang. Lalu mengekspor **200 sampel acak** (100 dipertahankan + 100 dibuang) ke `audit_filter_relevansi.csv` untuk dinilai manual.



In [ ]:
# Untuk mengunduh file hasil audit
from google.colab import files

# ===================== FILTER RELEVANSI =====================
# Pola regex kata kunci relevansi. HARUS SAMA dengan yang dipakai di notebook pemodelan
POLA_RELEVAN = (
    r'\b(?:tuli|tuna ?rungu|tunarungu|tunawicara|bisu|dongo|difabel|disabel|cacat|penyandang|isyarat|hrd|hasna|hasni|kembar)\b'
    r'|disabilit|wicara|kekurangan|keterbatasan|berkebutuhan|kerja|karyawan|pekerja|pegawai|kesempatan|lowongan|lamar'
    r'|diskriminasi|mandiri|kemampuan|nerima|menerima|diterima|melayani|ramah|sopan|sabar|tabah|prestasi|perusahaan|atasan|bos\b'
)

# True bila teks asli (huruf kecil) memuat salah satu kata kunci
df_final['relevan'] = df_final['text_casefold'].str.contains(POLA_RELEVAN, regex=True)

# Statistik sebelum/sesudah filter
n_sebelum = len(df_final)
print(f"Sebelum filter relevansi : {n_sebelum:,}")
print(f"Relevan (dipertahankan)  : {df_final['relevan'].sum():,} ({df_final['relevan'].mean():.1%})")
print(f"Tidak relevan (dibuang)  : {(~df_final['relevan']).sum():,}")
# Bandingkan sebaran label sebelum vs sesudah filter (perhatikan perubahan proporsi eksklusi)
print("\nDistribusi label SEBELUM filter:\n", df_final[LABEL_COL].value_counts().to_string())
print("\nDistribusi label SESUDAH filter:\n", df_final.loc[df_final['relevan'], LABEL_COL].value_counts().to_string())

# Ambil 100 komentar yang dipertahankan + 100 yang dibuang, lalu acak urutannya
# Sampel audit manual: 100 dipertahankan + 100 dibuang, diacak agar penilai tidak tahu statusnya
# Gabungkan dua sampel
audit = pd.concat([
    df_final[df_final['relevan']].sample(min(100, int(df_final['relevan'].sum())), random_state=RANDOM_STATE),
    df_final[~df_final['relevan']].sample(min(100, int((~df_final['relevan']).sum())), random_state=RANDOM_STATE),
]).sample(frac=1, random_state=RANDOM_STATE)
# Tambah kolom kosong 'penilaian_manual' untuk diisi: relevan / tidak_relevan
audit = audit.assign(status_filter=np.where(audit['relevan'], 'dipertahankan', 'dibuang'), penilaian_manual='')
audit[['text', 'penilaian_manual', 'status_filter']].to_csv('audit_filter_relevansi.csv', index=False)
# Unduh file audit
files.download('audit_filter_relevansi.csv')

# Buang komentar tidak relevan; HANYA yang lolos diteruskan ke model
# Hanya komentar relevan yang diteruskan ke tahap berikutnya (dan ke model)
# df_final sekarang berisi 2.955 komentar relevan
df_final = df_final[df_final['relevan']].drop(columns='relevan').reset_index(drop=True)
print(f"\nJumlah data final setelah filter relevansi: {len(df_final):,}")

Sebelum filter relevansi : 13,565
Relevan (dipertahankan)  : 2,955 (21.8%)
Tidak relevan (dibuang)  : 10,610

Distribusi label SEBELUM filter:
 label_annotator_1
eksklusi    7647
inklusi     5918

Distribusi label SESUDAH filter:
 label_annotator_1
inklusi     2124
eksklusi     831


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Jumlah data final setelah filter relevansi: 2,955


### HASIL PREPRO

### Sel 11 — Cek kualitas: token yang belum dikenal
 menghitung token yang tidak ada di kamus kata dasar, kamus formal, maupun stopword, lalu menampilkan 100 yang paling sering. daftar ini menunjukkan kata yang masih perlu dimasukkan ke kamus normalisasi (typo, bahasa daerah, nama orang).
**Cek hasil:** teratas biasanya nama (`kdm`, `hasni`, `hasna`, `store`, `dedi`), bukan kata bermakna persepsi. **Ini bukti model berisiko belajar nama video**, bukan sikap.
> Sel diagnostik: tidak mengubah data.

In [ ]:
# Kumpulan kata baku yang dikenal (kamus formal + hasil normalisasi)
KAMUS_FORMAL = set(kamus['formal'].dropna().astype(str)) | set(NORM.values())
# Hitung frekuensi tiap token setelah normalisasi
tok = Counter(w for ts in df_cleaned['tokens_norm'] for w in ts)
# OOV = token yang tidak dikenal kamus mana pun (tanpa stopword, > 2 huruf)
oov = [(w, c) for w, c in tok.items()
       if w not in ROOT_WORDS and w not in KAMUS_FORMAL and w not in STOP_BARU and len(w) > 2]
# Urutkan dari yang paling sering
oov.sort(key=lambda x: -x[1])
# Berapa token unik yang belum dikenal
print(f"Token unik belum dikenal: {len(oov):,} dari {len(tok):,}")
# Tampilkan 100 teratas
pd.DataFrame(oov[:100], columns=['token', 'frekuensi'])

Token unik belum dikenal: 8,840 dari 12,947


,token,frekuensi
0,kdm,1397
1,hasni,525
2,store,497
3,hasna,490
4,dedi,438
...,...,...
95,ibunya,30
96,berjaya,30
97,bali,29
98,pakuan,29


### Sel 12 — Cek kualitas: hasil stemming
**Apa yang dilakukan:** membandingkan kata sebelum dan sesudah stemming, urut berdasarkan frekuensi, dan menandai kata yang terpotong terlalu pendek (≥ 6 huruf menjadi ≤ 3 huruf).
 tinjau daftar; kata yang salah potong bisa ditambahkan ke `STEM_EXC_BARU` pada Sel 5.
> Sel diagnostik: tidak mengubah data.

In [ ]:
# Frekuensi kata sebelum stemming
sebelum = Counter(w for ts in df_cleaned['tokens_stop'] for w in ts)
# Pasangan (kata, hasil stem, frekuensi) untuk kata yang berubah
rows = [(w, stem_token(w), c) for w, c in sebelum.items() if stem_token(w) != w]
# Urutkan dari yang paling sering berubah
audit = pd.DataFrame(rows, columns=['kata', 'hasil_stem', 'frekuensi']).sort_values('frekuensi', ascending=False)
# Ringkasan: berapa kata berubah
print(f"Kata unik berubah: {len(audit):,} dari {len(sebelum):,}")
display(audit.head(100))
# Kata ≥ 6 huruf yang terpotong menjadi ≤ 3 huruf = kandidat salah-stem
print("\nDicurigai terpotong terlalu pendek (kata >= 6 huruf menjadi <= 3 huruf):")
display(audit[(audit['kata'].str.len() >= 6) & (audit['hasil_stem'].str.len() <= 3)].head(40))

Kata unik berubah: 3,388 dari 12,458


,kata,hasil_stem,frekuensi
117,menerima,terima,294
7,mengomong,omong,228
145,bekerja,kerja,226
36,terharu,haru,190
451,pemimpin,pimpin,164
...,...,...,...
108,berusaha,usaha,22
434,menikahi,nikah,22
201,mudahkan,mudah,22
196,suaminya,suami,22



Dicurigai terpotong terlalu pendek (kata >= 6 huruf menjadi <= 3 huruf):


,kata,hasil_stem,frekuensi
17,berdua,dua,126
243,apapun,apa,41
239,tuanya,tua,32
594,ibunya,ibu,30
794,duanya,dua,20
429,doakan,doa,16
156,berhak,hak,13
1741,kemauan,mau,11
518,berdoa,doa,11
515,mendoakan,doa,7


### Sel 13 — Hapus duplikat `clean_text`
 mencari komentar berbeda yang menjadi `clean_text` sama (mis. "Hasna dan Hasni" vs "hasna hasni"), menampilkan yang **labelnya bertentangan**, lalu menghapus duplikat yang labelnya sama. duplikat yang jatuh ke data latih *dan* data uji membuat skor evaluasi terlalu optimis (kebocoran data).
**Cek hasil:** ±47 baris kembar, 7 teks berlabel konflik, 21 duplikat dihapus → **2.934** baris.
> Baris dengan label konflik **belum** dibuang di sini; baru dibuang di notebook pemodelan (Tahap 2) → 2.920 baris.

In [ ]:
# Tandai baris yang clean_text-nya kembar (keep=False = tandai semuanya)
dup_mask = df_final.duplicated('clean_text', keep=False)
# Untuk tiap clean_text kembar: berapa jenis label berbeda
n_label = df_final[dup_mask].groupby('clean_text')[LABEL_COL].nunique()
# Daftar clean_text yang labelnya bertentangan (inklusi sekaligus eksklusi)
konflik = n_label[n_label > 1].index
# Statistik duplikasi
print(f"Baris dengan clean_text kembar : {dup_mask.sum():,}")
print(f"clean_text kembar beda label   : {len(konflik):,}")
# Tampilkan contoh konflik untuk ditinjau
display(df_final[df_final['clean_text'].isin(konflik)][['text', 'clean_text', LABEL_COL]]
        .sort_values('clean_text').head(30))

# Saklar: True = hapus duplikat (clean_text sama DAN label sama)
HAPUS_DUPLIKAT_CLEAN = True   # duplikat = clean_text sama DAN label sama
if HAPUS_DUPLIKAT_CLEAN:
    # Catat jumlah sebelum
    sebelum_n = len(df_final)
    # Sisakan satu per clean_text+label
    df_final = df_final.drop_duplicates(subset=['clean_text', LABEL_COL]).reset_index(drop=True)
    # Laporkan jumlah yang dihapus
    print(f"\nDuplikat dihapus: {sebelum_n - len(df_final):,} -> data final {len(df_final):,}")
# Sebaran label akhir
print(df_final[LABEL_COL].value_counts())

Baris dengan clean_text kembar : 47
clean_text kembar beda label   : 7


,text,clean_text,label_annotator_1
770,Alhamdulillah dapet kerjaan..,alhamdulillah dapat kerja,inklusi
728,Alhamdulillah dpt kerja,alhamdulillah dapat kerja,eksklusi
282,Cantik banget hasni,cantik banget hasni,eksklusi
478,Cantik banget mbk hasni,cantik banget hasni,inklusi
2368,Cantik hasna dan hasni,cantik hasna hasni,eksklusi
2509,Cantiknya hasna dan hasni.🥰🥰🥰,cantik hasna hasni,inklusi
353,Cantik nya Hasna Hasni,cantik hasna hasni,inklusi
2495,hasna❤,hasna,eksklusi
2334,Hasna ❤❤❤,hasna,inklusi
445,Hasna ♥️♥️♥️,hasna,eksklusi



Duplikat dihapus: 21 -> data final 2,934
label_annotator_1
inklusi     2117
eksklusi     817
Name: count, dtype: int64


### Sel 14 — Ringkasan dan simpan `data_preprocessed.csv`
mencetak statistik akhir (jumlah data, rata-rata token, ukuran kosakata) lalu menyimpan tiga kolom (`text`, `clean_text`, label) ke CSV dan mengunduhnya.
**Cek hasil:** 2.934 baris, rata-rata ±12 token/komentar, kosakata ±4.595 kata.

In [ ]:
# Untuk mengunduh file
from google.colab import files

# Jumlah kata per komentar
jml = df_final['clean_text'].str.split().str.len()
# Statistik akhir untuk bab hasil
print(f"Data final        : {len(df_final):,}")
print(f"Rata-rata token   : {jml.mean():.1f} | median {jml.median():.0f} | maks {jml.max()}")
print(f"Komentar 1-2 token: {(jml <= 2).sum():,} ({(jml <= 2).mean():.1%})")
print(f"Ukuran kosakata   : {len(set(w for t in df_final['clean_text'] for w in t.split())):,}")

# Nama file keluaran
NAMA_FILE = 'data_preprocessed.csv'
# Simpan: teks asli, teks bersih, label (3 kolom)
df_final[['text', 'clean_text', LABEL_COL]].to_csv(NAMA_FILE, index=False)
# Unduh ke komputer
files.download(NAMA_FILE)
print(f"Tersimpan dan diunduh: {NAMA_FILE}")

Data final        : 2,934
Rata-rata token   : 12.4 | median 9 | maks 332
Komentar 1-2 token: 64 (2.2%)
Ukuran kosakata   : 4,595


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Tersimpan dan diunduh: data_preprocessed.csv
